In [1]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout
from sklearn.model_selection import train_test_split

# -----------------------------
# 1. Load dataset
# -----------------------------
df = pd.read_csv("train.csv")

texts = df['comment_text'].astype(str).tolist()

# Multi-label targets (6 columns)
labels = df[['toxic','severe_toxic','obscene','threat','insult','identity_hate']].values

# -----------------------------
# 2. Tokenization
# -----------------------------
MAX_WORDS = 20000       # vocab size
MAX_SEQUENCE_LEN = 150  # sequence length

tokenizer = Tokenizer(num_words=MAX_WORDS, oov_token="<OOV>")
tokenizer.fit_on_texts(texts)
sequences = tokenizer.texts_to_sequences(texts)
X = pad_sequences(sequences, maxlen=MAX_SEQUENCE_LEN, padding='post', truncating='post')

y = labels

# -----------------------------
# 3. Train-Test Split
# -----------------------------
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

# -----------------------------
# 4. Build the Model
# -----------------------------
model = Sequential([
    Embedding(input_dim=MAX_WORDS+1, 
              output_dim=128, 
              input_length=MAX_SEQUENCE_LEN),
    LSTM(128, return_sequences=False),
    Dropout(0.5),
    Dense(64, activation='relu'),
    Dropout(0.3),
    Dense(6, activation='sigmoid')   # 6 outputs for multi-label classification
])

model.compile(loss='binary_crossentropy',
              optimizer='adam',
              metrics=['accuracy'])

model.summary()

# -----------------------------
# 5. Train the Model
# -----------------------------
history = model.fit(
    X_train, y_train,
    epochs=3,
    batch_size=128,
    validation_data=(X_val, y_val)
)

# -----------------------------
# 6. Evaluate
# -----------------------------
loss, acc = model.evaluate(X_val, y_val)
print(f"Validation Accuracy: {acc:.4f}")


/Users/midhunmanu/Developer/college/mini_project/.venv/lib/python3.11/site-packages/keras/src/layers/core/embedding.py:103: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

Epoch 1/3
998/998 ━━━━━━━━━━━━━━━━━━━━ 175s 175ms/step - accuracy: 0.7835 - loss: 0.1533 - val_accuracy: 0.9940 - val_loss: 0.1398
Epoch 2/3
998/998 ━━━━━━━━━━━━━━━━━━━━ 186s 187ms/step - accuracy: 0.9842 - loss: 0.1100 - val_accuracy: 0.9941 - val_loss: 0.0566
Epoch 3/3
998/998 ━━━━━━━━━━━━━━━━━━━━ 176s 176ms/step - accuracy: 0.9940 - loss: 0.0538 - val_accuracy: 0.9941 - val_loss: 0.0518
998/998 ━━━━━━━━━━━━━━━━━━━━ 26s 26ms/step - accuracy: 0.9941 - loss: 0.0518
Validation Accuracy: 0.9941


In [2]:
# -----------------------------
# 6. Make Predictions
# -----------------------------
def predict_comment(text, tokenizer, model, max_len=150):
    # Convert text -> sequence
    seq = tokenizer.texts_to_sequences([text])
    padded = pad_sequences(seq, maxlen=max_len, padding='post', truncating='post')
    
    # Predict
    pred = model.predict(padded)[0]
    
    # Map results to labels
    labels = ['toxic','severe_toxic','obscene','threat','insult','identity_hate']
    results = {label: float(round(p, 3)) for label, p in zip(labels, pred)}
    return results

# Example predictions
examples = [
    "I hate you, you are the worst!",
    "Have a great day my friend 😊",
    "I will find you and hurt you",
    "I Love You",
    "I will kill you"
]

for text in examples:
    print(f"\nComment: {text}")
    print(predict_comment(text, tokenizer, model))



Comment: I hate you, you are the worst!
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 95ms/step
{'toxic': 0.7720000147819519, 'severe_toxic': 0.03500000014901161, 'obscene': 0.3869999945163727, 'threat': 0.032999999821186066, 'insult': 0.38199999928474426, 'identity_hate': 0.07699999958276749}

Comment: Have a great day my friend 😊
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step
{'toxic': 0.05900000035762787, 'severe_toxic': 0.0, 'obscene': 0.00800000037997961, 'threat': 0.0010000000474974513, 'insult': 0.010999999940395355, 'identity_hate': 0.003000000026077032}

Comment: I will find you and hurt you
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step
{'toxic': 0.11100000143051147, 'severe_toxic': 0.0, 'obscene': 0.019999999552965164, 'threat': 0.003000000026077032, 'insult': 0.027000000700354576, 'identity_hate': 0.00800000037997961}

Comment: I Love You
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step
{'toxic': 0.029999999329447746, 'severe_toxic': 0.0, 'obscene': 0.004000000189989805, 'threat': 0.0, 'insult': 0.004999999888241291, '

model.save("toxic_comment.h5")

In [3]:
# -----------------------------
# 7. Save Model and Tokenizer for FastAPI
# -----------------------------
print("💾 Saving model and tokenizer...")

# Save the base model (NOT the adversarial one)
model.save("base_toxic_model.h5")

# Save tokenizer
import json
tokenizer_json = tokenizer.to_json()
with open('base_tokenizer.json', 'w', encoding='utf-8') as f:
    f.write(tokenizer_json)

print("✅ Base model saved as 'base_toxic_model.h5'")
print("✅ Tokenizer saved as 'base_tokenizer.json'")
print(f"📚 Vocabulary size: {len(tokenizer.word_index)}")

# Test it works
test_texts = ["I will kill you", "hello world", "fuck you"]
for text in test_texts:
    seq = tokenizer.texts_to_sequences([text])
    padded = pad_sequences(seq, maxlen=MAX_SEQUENCE_LEN, padding='post', truncating='post')
    pred = model.predict(padded, verbose=0)[0]
    print(f"'{text}' -> toxic: {pred[0]:.4f}")

💾 Saving model and tokenizer...
✅ Base model saved as 'base_toxic_model.h5'
✅ Tokenizer saved as 'base_tokenizer.json'
📚 Vocabulary size: 210338
'I will kill you' -> toxic: 0.7804
'hello world' -> toxic: 0.2474
'fuck you' -> toxic: 0.9926


In [4]:
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

# Class weights to handle imbalance
toxic_ratio = (y_train.sum(axis=1) > 0).mean()
class_weight = {0: 1 / (1 - toxic_ratio), 1: 1 / toxic_ratio}

# Callbacks
early_stop = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)
checkpoint = ModelCheckpoint('best_model.keras', save_best_only=True, monitor='val_loss')

history = model.fit(
    X_train, y_train,
    epochs=15,
    batch_size=256,
    validation_data=(X_val, y_val),
    class_weight=class_weight,
    callbacks=[early_stop, checkpoint]
)


Epoch 1/15
499/499 ━━━━━━━━━━━━━━━━━━━━ 125s 249ms/step - accuracy: 0.9937 - loss: 0.0511 - val_accuracy: 0.9941 - val_loss: 0.0514
Epoch 2/15
499/499 ━━━━━━━━━━━━━━━━━━━━ 140s 280ms/step - accuracy: 0.9927 - loss: 0.0467 - val_accuracy: 0.9941 - val_loss: 0.0517
Epoch 3/15
499/499 ━━━━━━━━━━━━━━━━━━━━ 148s 298ms/step - accuracy: 0.9894 - loss: 0.0439 - val_accuracy: 0.9941 - val_loss: 0.0556
Epoch 4/15
499/499 ━━━━━━━━━━━━━━━━━━━━ 130s 261ms/step - accuracy: 0.9892 - loss: 0.0411 - val_accuracy: 0.9941 - val_loss: 0.0574
